# 10 Topic Clustering, Keyword Co-occurrence Network, and Topic Matrix

This notebook handles Subagent 10's topic-clustering and keyword-network outputs. Inputs are fixed to the new master dataset and the 37-term keyword table:

- `data/NSFC正式增量采集_2014-2026_去重筛选最终结果.csv`
- `data/NSFC正式增量采集_37个关键词.csv`

Outputs are restricted to:

- `output/figures/Fig10_topic_network.svg/pdf/tiff/png`
- `output/tables/Table1_topic_summary_matrix.csv`
- `output/tables/10_keyword_edges.csv`
- `output/logs/10_topic_text.md`

Note: code comments and human-facing text are in English, and all visible figure text is English for use in the English manuscript figures.


## Chart Contract

- Analytical question: Which keyword co-occurrence communities organize BAE-related sensing, methods, objects, and performance outcomes in the deduplicated project corpus?
- Takeaway: The corpus separates into six interpretable BAE topic clusters, with AI-enabled urban form and land-use measurement as the largest dominant cluster.
- Chart family: Relationship / network visualization, rendered as a static Python figure.
- Data grain: One node per matched keyword from the 37-term keyword table; one edge per keyword pair co-occurring in at least one record; node size is record frequency and edge width is co-occurrence count.
- Density rule: The edge table keeps the complete weighted network, while the figure keeps high-frequency nodes and the strongest readable edges plus minimum incident edges for node context.
- Palette policy: Relaxed multi-category, using six low-saturation topic colors plus neutral edges.
- Export targets: SVG, PDF, TIFF, and PNG under `output/figures/`.

In [ ]:
# ===== 1. Environment Setup, Paths, and Style =====
# This notebook only reads the new master dataset and the 37-term keyword table; it does not write back to the data directory.
# Output logic is retained for later Fig10/table/log workers to rerun; this update does not execute export cells.

from pathlib import Path
from collections import Counter, defaultdict
from itertools import combinations
from datetime import datetime
import math
import os
import re
import textwrap

# In headless environments, give matplotlib a writable cache directory to avoid font-cache errors.
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import numpy as np
import pandas as pd
import matplotlib as mpl
from matplotlib import font_manager
import matplotlib.pyplot as plt

# Register and require Times New Roman; stop immediately if the font is unavailable instead of using a substitute.
TIMES_NEW_ROMAN_PATHS = [
    Path("/Library/Fonts/Times New Roman.ttf"),
    Path("/Library/Fonts/Times New Roman Bold.ttf"),
    Path("/Library/Fonts/Times New Roman Italic.ttf"),
    Path("/Library/Fonts/Times New Roman Bold Italic.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman Bold.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman Italic.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman Bold Italic.ttf"),
]
for font_path in TIMES_NEW_ROMAN_PATHS:
    if font_path.exists():
        font_manager.fontManager.addfont(str(font_path))
try:
    font_manager.findfont("Times New Roman", fallback_to_default=False)
except ValueError as exc:
    raise RuntimeError("Times New Roman is required for figure export but was not found by matplotlib.") from exc

from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import networkx as nx
import seaborn as sns
from IPython.display import display, Markdown


MAIN_DATASET_NAME = "NSFC正式增量采集_2014-2026_去重筛选最终结果.csv"
KEYWORD_TABLE_NAME = "NSFC正式增量采集_37个关键词.csv"
EXPECTED_RECORDS = 9222
EXPECTED_KEYWORD_COUNT = 37


def find_project_root(start: Path) -> Path:
    """Find the project root upward from the current directory so the notebook can run from code/ or the project root."""
    start = start.resolve()
    required_inputs = [MAIN_DATASET_NAME, KEYWORD_TABLE_NAME]
    for candidate in [start, *start.parents]:
        data_dir = candidate / "data"
        if all((data_dir / name).exists() for name in required_inputs):
            return candidate
    expected = ", ".join(f"data/{name}" for name in required_inputs)
    raise FileNotFoundError(f"Cannot find required inputs from the current working directory: {expected}")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_PATH = PROJECT_ROOT / "data" / MAIN_DATASET_NAME
KEYWORD_TABLE_PATH = PROJECT_ROOT / "data" / KEYWORD_TABLE_NAME
FIGURE_DIR = PROJECT_ROOT / "output" / "figures"
TABLE_DIR = PROJECT_ROOT / "output" / "tables"
LOG_DIR = PROJECT_ROOT / "output" / "logs"
CODE_PATH = PROJECT_ROOT / "code" / "10_topic_clustering_and_keyword_network.ipynb"

FIG_BASE = FIGURE_DIR / "Fig10_topic_network"
SUMMARY_TABLE_PATH = TABLE_DIR / "Table1_topic_summary_matrix.csv"
EDGE_TABLE_PATH = TABLE_DIR / "10_keyword_edges.csv"
LOG_PATH = LOG_DIR / "10_topic_text.md"

for folder in [FIGURE_DIR, TABLE_DIR, LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Base style for static figures: all visible figure text is English and the font is Times New Roman serif.
mpl.rcParams.update({
    "figure.facecolor": "#FFFFFF",
    "axes.facecolor": "#FFFFFF",
    "savefig.facecolor": "#FFFFFF",
    "savefig.edgecolor": "none",
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "font.sans-serif": ["Times New Roman"],
    "font.monospace": ["Times New Roman"],
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

TOKENS = {
    "surface": "#FFFFFF",
    "panel": "#FFFFFF",
    "ink": "#1F2430",
    "muted": "#6F768A",
    "grid": "#E6E8F0",
    "axis": "#D7DBE7",
    "neutral_light": "#E2E5EA",
    "neutral_base": "#C5CAD3",
    "neutral_mid": "#7A828F",
    "neutral_dark": "#464C55",
}

THEME_COLORS = {
    "Remote sensing and urban thermal environment": {"fill": "#A3BEFA", "edge": "#2E4780", "label": "Remote sensing & thermal"},
    "Street-view and human-scale built environment": {"fill": "#F5BACC", "edge": "#8A3A6F", "label": "Street-view & human scale"},
    "Nighttime light and carbon-energy performance": {"fill": "#FFE15B", "edge": "#736422", "label": "Nighttime light & carbon-energy"},
    "Multisource data and urban resilience": {"fill": "#A3D576", "edge": "#386411", "label": "Multisource & resilience"},
    "AI-enabled urban form and land-use measurement": {"fill": "#F0986E", "edge": "#804126", "label": "AI urban form & land use"},
    "Exposure, health and environmental risk": {"fill": "#CEDFFE", "edge": "#5477C4", "label": "Exposure & risk"},
}

THEME_ORDER = list(THEME_COLORS.keys())
print(f"Project root: {PROJECT_ROOT}")
print(f"Input data: {DATA_PATH}")
print(f"Keyword table: {KEYWORD_TABLE_PATH}")

In [ ]:
# ===== 2. Load Data and Keyword Table, Then Parse Matched Terms =====
# The four matched_* fields contain semicolon-delimited Chinese labels; normalize Chinese semicolons, enumeration commas, and pipes here.
# The keyword network and topic matrix use only terms from the 37-term keyword table.

df = pd.read_csv(DATA_PATH)
keyword_df = pd.read_csv(KEYWORD_TABLE_PATH)

if len(df) != EXPECTED_RECORDS:
    raise ValueError(f"Expected {EXPECTED_RECORDS:,} records in the new main dataset, got {len(df):,}.")

required_keyword_columns = {"term", "term_group"}
missing_keyword_columns = required_keyword_columns - set(keyword_df.columns)
if missing_keyword_columns:
    raise ValueError(f"Missing required keyword-table columns: {sorted(missing_keyword_columns)}")

keyword_df = keyword_df.loc[:, ["term", "term_group"]].copy()
keyword_df["term"] = keyword_df["term"].astype(str).str.strip()
keyword_df["term_group"] = keyword_df["term_group"].astype(str).str.strip()
keyword_df = keyword_df.loc[keyword_df["term"].ne("")].drop_duplicates("term", keep="first").reset_index(drop=True)
if len(keyword_df) != EXPECTED_KEYWORD_COUNT:
    raise ValueError(f"Expected {EXPECTED_KEYWORD_COUNT} keyword terms, got {len(keyword_df)}.")

KEYWORD_TERMS = keyword_df["term"].tolist()
KEYWORD_TERM_SET = set(KEYWORD_TERMS)
KEYWORD_ORDER = {term: idx for idx, term in enumerate(KEYWORD_TERMS)}
KEYWORD_GROUPS = dict(zip(keyword_df["term"], keyword_df["term_group"]))

TERM_COLUMNS = [
    "matched_keywords",
    "matched_method_terms",
    "matched_object_terms",
    "matched_performance_terms",
]
REQUIRED_COLUMNS = [
    "project_title",
    "abstract_text",
    "keywords_raw",
    "outcomes_text",
    *TERM_COLUMNS,
]
missing_columns = [col for col in REQUIRED_COLUMNS if col not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")


def split_terms(value):
    """Split semicolon-delimited matched terms in one cell into a pre-deduplication list."""
    if pd.isna(value):
        return []
    text = str(value).replace("；", ";").replace("、", ";").replace("|", ";")
    terms = []
    for term in text.split(";"):
        term = term.strip()
        if term and term.lower() != "nan":
            terms.append(term)
    return terms


raw_term_frequency = Counter()
for col in TERM_COLUMNS:
    for value in df[col]:
        raw_term_frequency.update(split_terms(value))

unexpected_terms = sorted(set(raw_term_frequency) - KEYWORD_TERM_SET)
if unexpected_terms:
    raise ValueError(f"Matched fields contain terms outside the 37-term keyword table: {unexpected_terms}")

missing_keyword_terms = sorted(KEYWORD_TERM_SET - set(raw_term_frequency), key=KEYWORD_ORDER.__getitem__)
if missing_keyword_terms:
    raise ValueError(f"Keyword-table terms not found in the main dataset matched fields: {missing_keyword_terms}")


def record_terms(row):
    """Collect unique keywords from the four matched fields in one record and sort them by keyword-table order."""
    terms = set()
    for col in TERM_COLUMNS:
        terms.update(term for term in split_terms(row[col]) if term in KEYWORD_TERM_SET)
    return sorted(terms, key=KEYWORD_ORDER.__getitem__)


# Append in-memory fields to each record without writing back to the original CSV.
df = df.copy()
df["_terms"] = df.apply(record_terms, axis=1)
df["_term_count"] = df["_terms"].map(len)

term_frequency = Counter()
for terms in df["_terms"]:
    term_frequency.update(terms)

# Determine term roles only from method/object/performance fields to avoid role noise from the aggregate matched_keywords field.
role_counts = defaultdict(Counter)
role_column_map = {
    "method": "matched_method_terms",
    "object": "matched_object_terms",
    "performance": "matched_performance_terms",
}
for _, row in df.iterrows():
    for role, col in role_column_map.items():
        for term in set(split_terms(row[col])):
            if term in KEYWORD_TERM_SET:
                role_counts[term][role] += 1


def dominant_role(term):
    """Return the most common role for a keyword across method/object/performance fields."""
    counts = role_counts.get(term, Counter())
    if not counts:
        return "mixed"
    top = counts.most_common()
    if len(top) > 1 and top[0][1] == top[1][1]:
        return "mixed"
    return top[0][0]


print(f"Records: {len(df):,}")
print(f"Keyword universe: {len(KEYWORD_TERMS):,} terms from {KEYWORD_TABLE_PATH.name}")
print(f"Unique matched terms used: {len(term_frequency):,}")
print("Top matched terms:")
for term, count in term_frequency.most_common(12):
    print(f"- {term}: {count}")


In [ ]:
# ===== 3. Topic-Cluster Definitions and Record-Level Primary Topic Assignment =====
# Use interpretable rule-based topic assignment: each topic has a weighted set of seed terms.
# Higher-weight terms indicate stronger topical direction; broad terms receive lower weights so they do not determine topics on their own.

THEME_WEIGHTS = {
    "Remote sensing and urban thermal environment": {
        "遥感": 3.0, "高分": 3.0, "热岛": 3.0, "热环境": 2.5,
        "气候": 1.4, "GIS": 1.0, "地理信息": 1.0, "绿地": 1.5,
        "生态环境": 1.0, "城市群": 0.6,
    },
    "Street-view and human-scale built environment": {
        "街景": 4.0, "街道": 3.0, "街区": 3.0, "POI": 3.0,
        "LBS": 3.0, "手机信令": 3.0, "轨迹": 3.0, "建成环境": 2.2,
        "大数据": 0.9, "规划": 0.5,
    },
    "Nighttime light and carbon-energy performance": {
        "夜间灯光": 4.0, "碳": 3.2, "能源": 3.2, "建筑": 1.5,
        "规划": 0.7, "大数据": 0.8, "遥感": 0.8, "高分": 0.8,
    },
    "Multisource data and urban resilience": {
        "多源数据": 3.2, "韧性": 3.0, "洪涝": 3.0, "海绵": 3.0,
        "生态环境": 2.0, "气候": 1.2, "GIS": 1.0, "遥感": 1.0,
        "城市群": 0.8,
    },
    "AI-enabled urban form and land-use measurement": {
        "人工智能": 4.0, "机器学习": 3.5, "深度学习": 3.5, "三维": 3.0,
        "LiDAR": 3.0, "城市形态": 3.0, "土地利用": 2.4, "规划": 1.7,
        "城市群": 1.3, "大数据": 1.6, "GIS": 1.2, "建筑": 1.0,
    },
    "Exposure, health and environmental risk": {
        "暴露": 4.0, "空气污染": 4.0, "生态环境": 1.6, "热环境": 1.8,
        "绿地": 1.5, "建筑": 1.0, "气候": 1.0, "洪涝": 1.0,
        "热岛": 0.8,
    },
}


def topic_scores(terms):
    """Compute one record's scores across the six topics."""
    terms = set(terms)
    scores = {}
    specific_hits = {}
    for theme in THEME_ORDER:
        weights = THEME_WEIGHTS[theme]
        scores[theme] = sum(weights.get(term, 0.0) for term in terms)
        specific_hits[theme] = sum(1 for term in terms if weights.get(term, 0.0) >= 2.5)
    return scores, specific_hits


def assign_topic(terms):
    """Assign the primary topic by topic score, count of strong-direction terms, and fixed topic order."""
    scores, specific_hits = topic_scores(terms)
    best_theme = max(
        THEME_ORDER,
        key=lambda theme: (scores[theme], specific_hits[theme], -THEME_ORDER.index(theme)),
    )
    if scores[best_theme] <= 0:
        # This should not occur in this dataset; fall back to the broadest urban-form measurement topic.
        best_theme = "AI-enabled urban form and land-use measurement"
    return best_theme


df["topic_cluster"] = df["_terms"].map(assign_topic)
cluster_counts = df["topic_cluster"].value_counts().reindex(THEME_ORDER, fill_value=0)
cluster_share = (cluster_counts / len(df) * 100).round(1)

cluster_overview = pd.DataFrame({
    "Topic cluster": cluster_counts.index,
    "N": cluster_counts.values,
    "Share (%)": cluster_share.values,
})
display(cluster_overview)

# Ensure all six topics have samples as required.
assert len(cluster_counts) == 6
assert (cluster_counts > 0).all(), "All six topic clusters should contain at least one record."
print("Largest cluster:", cluster_counts.idxmax(), int(cluster_counts.max()))

In [ ]:
# ===== 4. Build the Keyword Co-occurrence Edge Table =====
# The complete edge table keeps all keyword pairs; the figure applies separate readability filtering.

TERM_LABELS = {
    "遥感": "Remote sensing",
    "大数据": "Big data",
    "机器学习": "Machine learning",
    "手机信令": "Mobile signaling",
    "高分": "High-resolution imagery",
    "建成环境": "Built environment",
    "城市形态": "Urban morphology",
    "规划": "Planning",
    "碳": "Carbon",
    "能源": "Energy",
    "GIS": "GIS",
    "土地利用": "Land use",
    "气候": "Climate",
    "城市群": "Urban agglomeration",
    "生态环境": "Ecological environment",
    "夜间灯光": "Nighttime light",
    "建筑": "Buildings",
    "绿地": "Green space",
    "街景": "Street view",
    "POI": "POI",
    "轨迹": "Mobility traces",
    "街区": "Neighborhood/block",
    "多源数据": "Multisource data",
    "三维": "3D modeling",
    "深度学习": "Deep learning",
    "热岛": "Urban heat island",
    "韧性": "Resilience",
    "热环境": "Thermal environment",
    "暴露": "Exposure",
    "地理信息": "Geoinformation",
    "洪涝": "Flooding",
    "街道": "Street network",
    "海绵": "Sponge city",
    "人工智能": "Artificial intelligence",
    "LiDAR": "LiDAR",
    "LBS": "LBS",
    "空气污染": "Air pollution",
}

# Every term in the 37-term keyword table must have an English legend label to keep Chinese text out of the figure.
missing_labels = sorted(KEYWORD_TERM_SET - set(TERM_LABELS), key=KEYWORD_ORDER.__getitem__)
if missing_labels:
    raise ValueError(f"Missing English labels for keyword-table terms: {missing_labels}")
extra_labels = sorted(set(TERM_LABELS) - KEYWORD_TERM_SET)
if extra_labels:
    raise ValueError(f"TERM_LABELS contains terms outside the keyword table: {extra_labels}")


def term_topic(term):
    """Assign each keyword to the highest-weight topic for node coloring."""
    return max(
        THEME_ORDER,
        key=lambda theme: (THEME_WEIGHTS[theme].get(term, 0.0), -THEME_ORDER.index(theme)),
    )


term_topic_map = {term: term_topic(term) for term in KEYWORD_TERMS if term in term_frequency}
edge_counter = Counter()
for terms in df["_terms"]:
    for source, target in combinations(sorted(set(terms)), 2):
        edge_counter[(source, target)] += 1

edge_rows = []
for (source, target), weight in edge_counter.items():
    source_freq = term_frequency[source]
    target_freq = term_frequency[target]
    union_count = source_freq + target_freq - weight
    jaccard = weight / union_count if union_count else 0
    source_topic = term_topic_map[source]
    target_topic = term_topic_map[target]
    edge_rows.append({
        "source_term": source,
        "target_term": target,
        "source_label": TERM_LABELS[source],
        "target_label": TERM_LABELS[target],
        "source_term_group": KEYWORD_GROUPS[source],
        "target_term_group": KEYWORD_GROUPS[target],
        "cooccurrence_count": weight,
        "source_frequency": source_freq,
        "target_frequency": target_freq,
        "jaccard_weight": round(jaccard, 4),
        "source_topic_cluster": source_topic,
        "target_topic_cluster": target_topic,
        "edge_scope": "within_cluster" if source_topic == target_topic else "between_clusters",
    })

edge_df = pd.DataFrame(edge_rows).sort_values(
    ["cooccurrence_count", "jaccard_weight", "source_label", "target_label"],
    ascending=[False, False, True, True],
).reset_index(drop=True)

edge_df.to_csv(EDGE_TABLE_PATH, index=False, encoding="utf-8-sig")
print(f"Complete weighted edge table: {EDGE_TABLE_PATH}")
print(f"Edges: {len(edge_df):,}")
display(edge_df.head(12))

In [ ]:
# ===== 5. Generate Results-Oriented Topic Summary Table 1 =====
# Table 1 reports empirical topic structure rather than repeating the Fig. 3 coding framework.

OBJECT_LABELS = {term: TERM_LABELS[term] for term in [
    "建成环境", "城市形态", "规划", "土地利用", "城市群", "建筑", "绿地", "街区", "街道", "海绵"
]}
PERFORMANCE_LABELS = {
    "热岛": "urban heat island",
    "热环境": "thermal environment",
    "碳": "carbon emissions",
    "能源": "energy performance",
    "空气污染": "air pollution",
    "暴露": "exposure",
    "韧性": "resilience",
    "洪涝": "flooding",
    "气候": "climate adaptation",
    "生态环境": "ecological performance",
}
EMPIRICAL_INTERPRETATIONS = {
    "Remote sensing and urban thermal environment": "Regional remote-sensing studies connect land-use, planning and green-space objects with climate, carbon, ecological and heat outcomes.",
    "Street-view and human-scale built environment": "Human-scale mobility and street-network work links planning, building and block objects to climate, resilience and carbon endpoints.",
    "Nighttime light and carbon-energy performance": "Carbon-energy studies concentrate on planning and building objects, with carbon and energy outcomes dominating the topic evidence.",
    "Multisource data and urban resilience": "Multisource resilience work ties planning, sponge-city and land-use objects to climate, resilience, ecology and flooding outcomes.",
    "AI-enabled urban form and land-use measurement": "AI and 3D measurement studies translate planning, building, land-use and morphology objects into carbon, climate and energy assessments.",
    "Exposure, health and environmental risk": "Risk-oriented studies link building, planning, land-use and green-space objects to exposure, air pollution and climate endpoints.",
}


def top_join(counter, mapping=None, top_n=5):
    """Convert the top N Counter entries into English phrases; return keys directly when mapping is empty."""
    items = []
    for key, count in counter.most_common():
        label = mapping.get(key, key) if mapping else key
        if label not in items:
            items.append(label)
        if len(items) >= top_n:
            break
    return "; ".join(items) if items else "not specified"


def count_terms(records, allowed_terms=None):
    """Count keywords by record-level occurrence to avoid inflating repeated labels within the same record."""
    counter = Counter()
    allowed = set(allowed_terms) if allowed_terms is not None else None
    for terms in records:
        for term in set(terms):
            if allowed is None or term in allowed:
                counter[term] += 1
    return counter


def count_field_terms(part, column, allowed_terms):
    """Recompute dominant objects and performance endpoints within a topic from matched_object_terms / matched_performance_terms."""
    return count_terms((split_terms(value) for value in part[column]), allowed_terms=allowed_terms)


def join_ordered(keys, mapping, top_n):
    """Output deduplicated labels in the already sorted key order."""
    items = []
    for key in keys:
        label = mapping.get(key, key)
        if label not in items:
            items.append(label)
        if len(items) >= top_n:
            break
    return "; ".join(items) if items else "not specified"


def top_weighted_keywords(counter, theme, top_n=6):
    """Select representative keywords by topic weight times within-topic record count."""
    weights = THEME_WEIGHTS[theme]
    ranked_keys = [
        term
        for term, count in sorted(
            counter.items(),
            key=lambda item: (item[1] * weights.get(item[0], 0.35), item[1], -KEYWORD_ORDER[item[0]]),
            reverse=True,
        )
    ]
    return join_ordered(ranked_keys, TERM_LABELS, top_n=top_n)


summary_rows = []
for idx, theme in enumerate(THEME_ORDER, start=1):
    part = df.loc[df["topic_cluster"] == theme].copy()
    all_term_counter = count_terms(part["_terms"])
    object_counter = count_field_terms(part, "matched_object_terms", OBJECT_LABELS.keys())
    performance_counter = count_field_terms(part, "matched_performance_terms", PERFORMANCE_LABELS.keys())
    summary_rows.append({
        "topic_id": f"T{idx}",
        "topic_label": theme,
        "n_records": int(len(part)),
        "share_percent": round(len(part) / len(df) * 100, 1),
        "representative_keywords": top_weighted_keywords(all_term_counter, theme, top_n=6),
        "main_built_environment_objects": top_join(object_counter, OBJECT_LABELS, top_n=5),
        "main_performance_endpoints": top_join(performance_counter, PERFORMANCE_LABELS, top_n=5),
        "empirical_interpretation": EMPIRICAL_INTERPRETATIONS[theme],
    })

summary_df = pd.DataFrame(summary_rows)
expected_topic_counts = [2303, 577, 2713, 410, 2803, 416]
if summary_df["n_records"].tolist() != expected_topic_counts:
    raise ValueError(f"Topic counts changed: {summary_df['n_records'].tolist()}")
if int(summary_df["n_records"].sum()) != EXPECTED_RECORDS:
    raise ValueError("Table1 n_records do not sum to 9222.")

summary_df.to_csv(SUMMARY_TABLE_PATH, index=False, encoding="utf-8-sig")
print(f"Empirical topic summary matrix: {SUMMARY_TABLE_PATH}")
display(summary_df)


In [ ]:
# ===== 6. Draw Fig10: Single-Panel Topic Keyword Network =====
# Combine six topic centers, aggregate inter-topic connections, and local keyword subnetworks into one single-panel figure.

import matplotlib.patheffects as pe
from matplotlib.patches import FancyArrowPatch

FIG10_THEME_COLORS = {
    "Remote sensing and urban thermal environment": {"fill": "#AFC7E8", "edge": "#3D5F93", "label": THEME_COLORS["Remote sensing and urban thermal environment"]["label"]},
    "Street-view and human-scale built environment": {"fill": "#DDA6BF", "edge": "#8A4E70", "label": THEME_COLORS["Street-view and human-scale built environment"]["label"]},
    "Nighttime light and carbon-energy performance": {"fill": "#E7D37E", "edge": "#8C7A2A", "label": THEME_COLORS["Nighttime light and carbon-energy performance"]["label"]},
    "Multisource data and urban resilience": {"fill": "#B9D7A1", "edge": "#4E7A35", "label": THEME_COLORS["Multisource data and urban resilience"]["label"]},
    "AI-enabled urban form and land-use measurement": {"fill": "#E7A07D", "edge": "#93513A", "label": THEME_COLORS["AI-enabled urban form and land-use measurement"]["label"]},
    "Exposure, health and environmental risk": {"fill": "#C7D7F2", "edge": "#5E75A8", "label": THEME_COLORS["Exposure, health and environmental risk"]["label"]},
}

THEME_LINK_COLOR = "#C7352F"
KEYWORD_LINK_COLOR = "#8E96A3"
NODE_MIN_FREQUENCY = 7
EDGE_MIN_WEIGHT = 8
MAX_PLOTTED_EDGES = 70
GLOBAL_BACKBONE_EDGES = 30
BRIDGE_BACKBONE_EDGES = 18
KEYWORD_TOP_EDGES_PER_NODE = 1


def wrap_label(label, width=18):
    return "\n".join(textwrap.wrap(label, width=width, break_long_words=False))


def scale_width(value, values, min_width=0.55, max_width=4.2):
    values = np.asarray(list(values), dtype=float)
    if values.size == 0 or values.max() == values.min():
        return (min_width + max_width) / 2
    value = float(value)
    lo, hi = math.sqrt(values.min()), math.sqrt(values.max())
    return min_width + (max_width - min_width) * (math.sqrt(value) - lo) / (hi - lo)


def draw_curved_link(ax, source_xy, target_xy, width, color, alpha, rad, zorder):
    patch = FancyArrowPatch(
        source_xy,
        target_xy,
        arrowstyle="-",
        connectionstyle=f"arc3,rad={rad}",
        linewidth=width,
        color=color,
        alpha=alpha,
        mutation_scale=1,
        capstyle="round",
        joinstyle="round",
        zorder=zorder,
    )
    ax.add_patch(patch)


# Aggregate complete term-level co-occurrence to topic-center edges.
cluster_graph = nx.Graph()
for theme in THEME_ORDER:
    cluster_graph.add_node(
        theme,
        n=int(cluster_counts.loc[theme]),
        label=FIG10_THEME_COLORS[theme]["label"],
    )

for _, row in edge_df.loc[edge_df["edge_scope"].eq("between_clusters")].iterrows():
    source_theme = row["source_topic_cluster"]
    target_theme = row["target_topic_cluster"]
    if source_theme == target_theme:
        continue
    previous = cluster_graph.get_edge_data(source_theme, target_theme, default={"weight": 0, "keyword_edges": 0})
    cluster_graph.add_edge(
        source_theme,
        target_theme,
        weight=previous["weight"] + int(row["cooccurrence_count"]),
        keyword_edges=previous["keyword_edges"] + 1,
    )

# Select a readable keyword backbone from the complete edge table.
visible_nodes = {term for term, count in term_frequency.items() if count >= NODE_MIN_FREQUENCY}
candidate_edges = edge_df.loc[
    edge_df["source_term"].isin(visible_nodes) & edge_df["target_term"].isin(visible_nodes)
].copy()

selected_edge_lookup = {}


def add_keyword_edge(row):
    source = row["source_term"]
    target = row["target_term"]
    key = tuple(sorted((source, target)))
    weight = int(row["cooccurrence_count"])
    current = selected_edge_lookup.get(key)
    if current is None or weight > current["weight"]:
        selected_edge_lookup[key] = {
            "source": source,
            "target": target,
            "weight": weight,
            "jaccard": float(row["jaccard_weight"]),
            "scope": row["edge_scope"],
        }

for _, row in candidate_edges.loc[candidate_edges["cooccurrence_count"] >= EDGE_MIN_WEIGHT].head(GLOBAL_BACKBONE_EDGES).iterrows():
    add_keyword_edge(row)

for _, row in candidate_edges.loc[candidate_edges["edge_scope"].eq("between_clusters")].head(BRIDGE_BACKBONE_EDGES).iterrows():
    add_keyword_edge(row)

for node in sorted(visible_nodes):
    incident = candidate_edges.loc[
        candidate_edges["source_term"].eq(node) | candidate_edges["target_term"].eq(node)
    ].head(KEYWORD_TOP_EDGES_PER_NODE)
    for _, row in incident.iterrows():
        add_keyword_edge(row)

selected_edges = sorted(
    selected_edge_lookup.values(),
    key=lambda item: (item["weight"], item["jaccard"]),
    reverse=True,
)[:MAX_PLOTTED_EDGES]

G = nx.Graph()
for term in sorted(visible_nodes):
    G.add_node(
        term,
        label=TERM_LABELS[term],
        frequency=term_frequency[term],
        topic=term_topic_map[term],
        role=dominant_role(term),
    )
for edge in selected_edges:
    G.add_edge(
        edge["source"],
        edge["target"],
        weight=edge["weight"],
        jaccard=edge["jaccard"],
        scope=edge["scope"],
    )

keyword_theme_centers = {
    "Remote sensing and urban thermal environment": np.array([-0.60, 0.62]),
    "Multisource data and urban resilience": np.array([-0.99, 0.02]),
    "Street-view and human-scale built environment": np.array([-0.76, -0.63]),
    "Exposure, health and environmental risk": np.array([0.09, -0.86]),
    "Nighttime light and carbon-energy performance": np.array([0.94, -0.58]),
    "AI-enabled urban form and land-use measurement": np.array([0.87, 0.39]),
}

keyword_pos = {}
theme_keyword_nodes = {}
for theme in THEME_ORDER:
    nodes = [node for node in G.nodes if G.nodes[node]["topic"] == theme]
    nodes = sorted(nodes, key=lambda node: (-G.nodes[node]["frequency"], TERM_LABELS[node]))
    theme_keyword_nodes[theme] = nodes
    if not nodes:
        continue
    center = keyword_theme_centers[theme]
    radius = 0.34 + 0.020 * min(len(nodes), 10)
    start_angle = math.radians(18 + 23 * THEME_ORDER.index(theme))
    for i, node in enumerate(nodes):
        angle = start_angle + 2 * math.pi * i / max(len(nodes), 1)
        jitter = 0.042 * np.array([math.cos(2.4 * angle), math.sin(2.1 * angle)])
        keyword_pos[node] = center + radius * np.array([math.cos(angle), math.sin(angle)]) + jitter

LAYOUT_X_SCALE = 0.84
LAYOUT_Y_SCALE = 1.48
keyword_theme_centers = {
    theme: np.array([center[0] * LAYOUT_X_SCALE, center[1] * LAYOUT_Y_SCALE])
    for theme, center in keyword_theme_centers.items()
}
keyword_pos = {
    node: np.array([position[0] * LAYOUT_X_SCALE, position[1] * LAYOUT_Y_SCALE])
    for node, position in keyword_pos.items()
}

fig, ax = plt.subplots(figsize=(12.2, 10.7), facecolor=TOKENS["surface"])
ax.set_facecolor(TOKENS["surface"])
ax.axis("off")

# Theme-center links use the aggregated cross-topic co-occurrence weights.
cluster_edge_weights = [data["weight"] for _, _, data in cluster_graph.edges(data=True)]
for edge_index, (source, target, data) in enumerate(sorted(cluster_graph.edges(data=True), key=lambda item: item[2]["weight"])):
    rad = 0.12 if edge_index % 2 == 0 else -0.12
    width = scale_width(data["weight"], cluster_edge_weights, min_width=1.05, max_width=6.4)
    alpha = min(0.58, 0.20 + 0.38 * data["weight"] / max(cluster_edge_weights))
    draw_curved_link(
        ax,
        keyword_theme_centers[source],
        keyword_theme_centers[target],
        width=width,
        color=THEME_LINK_COLOR,
        alpha=alpha,
        rad=rad,
        zorder=0,
    )

# Light radial spokes connect topic centers to their assigned keywords.
for theme, nodes in theme_keyword_nodes.items():
    center = keyword_theme_centers[theme]
    for node in nodes:
        x, y = keyword_pos[node]
        ax.plot(
            [center[0], x], [center[1], y],
            color=KEYWORD_LINK_COLOR,
            linewidth=0.95,
            alpha=0.24,
            zorder=1,
        )

# Keyword co-occurrence edges remain lighter than the theme-center links.
keyword_edge_weights = [data["weight"] for _, _, data in G.edges(data=True)]
for edge_index, (source, target, data) in enumerate(sorted(G.edges(data=True), key=lambda item: item[2]["weight"])):
    source_topic = G.nodes[source]["topic"]
    target_topic = G.nodes[target]["topic"]
    same_topic = source_topic == target_topic
    edge_color = KEYWORD_LINK_COLOR
    alpha = 0.44 if same_topic else 0.36
    draw_curved_link(
        ax,
        keyword_pos[source],
        keyword_pos[target],
        width=scale_width(data["weight"], keyword_edge_weights, min_width=0.55, max_width=2.75),
        color=edge_color,
        alpha=alpha,
        rad=0.05 if edge_index % 2 == 0 else -0.05,
        zorder=2,
    )

for theme in THEME_ORDER:
    center = keyword_theme_centers[theme]
    colors = FIG10_THEME_COLORS[theme]
    center_size = 1200 + 1.35 * int(cluster_counts.loc[theme])
    ax.scatter(
        [center[0]], [center[1]],
        s=center_size,
        facecolor=colors["fill"],
        edgecolor=colors["edge"],
        linewidth=1.35,
        alpha=0.91,
        zorder=5,
    )
    ax.text(
        center[0], center[1], wrap_label(colors["label"], width=15),
        ha="center", va="center", fontsize=10.8, fontweight="bold",
        color=colors["edge"], linespacing=0.84, zorder=7,
    )

for theme in THEME_ORDER:
    nodes = [node for node in G.nodes if G.nodes[node]["topic"] == theme]
    if not nodes:
        continue
    frequencies = np.array([G.nodes[node]["frequency"] for node in nodes])
    sizes = 70 + 16.5 * np.sqrt(frequencies)
    nx.draw_networkx_nodes(
        G,
        keyword_pos,
        nodelist=nodes,
        node_size=sizes,
        node_color=FIG10_THEME_COLORS[theme]["fill"],
        edgecolors=FIG10_THEME_COLORS[theme]["edge"],
        linewidths=1.0,
        alpha=0.97,
        ax=ax,
    )

for node, (x, y) in keyword_pos.items():
    theme = G.nodes[node]["topic"]
    center = keyword_theme_centers[theme]
    vec = np.array([x, y]) - center
    norm = np.linalg.norm(vec)
    if norm < 0.05:
        dx, dy, ha, va = 0.0, 0.13, "center", "bottom"
    else:
        direction = vec / norm
        dx = 0.065 * direction[0]
        dy = 0.065 * direction[1]
        ha = "left" if direction[0] > 0.20 else "right" if direction[0] < -0.20 else "center"
        va = "bottom" if direction[1] > 0.35 else "top" if direction[1] < -0.35 else "center"
    freq = G.nodes[node]["frequency"]
    ax.text(
        x + dx,
        y + dy,
        wrap_label(G.nodes[node]["label"], width=15),
        ha=ha,
        va=va,
        fontsize=10.8 if freq >= 35 else 10.2,
        fontweight="semibold" if freq >= 55 else "normal",
        color=TOKENS["ink"],
        linespacing=0.88,
        zorder=8,
        path_effects=[pe.withStroke(linewidth=3.8, foreground=TOKENS["surface"], alpha=0.97)],
    )

ax.set_xlim(-1.30, 1.30)
ax.set_ylim(-2.15, 2.15)

legend_handles = [
    Patch(
        facecolor=FIG10_THEME_COLORS[theme]["fill"],
        edgecolor=FIG10_THEME_COLORS[theme]["edge"],
        label=FIG10_THEME_COLORS[theme]["label"],
    )
    for theme in THEME_ORDER
]
fig.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.43, 0.185),
    ncol=3,
    frameon=False,
    fontsize=12.8,
    handlelength=1.55,
    columnspacing=1.70,
    handletextpad=0.58,
    labelspacing=1.04,
)
line_handles = [
    Line2D([0], [0], color=THEME_LINK_COLOR, lw=5.8, alpha=0.82, label="Topic links"),
    Line2D([0], [0], color=KEYWORD_LINK_COLOR, lw=3.0, alpha=0.78, label="Keyword co-occurrence"),
]
fig.legend(
    handles=line_handles,
    loc="lower right",
    bbox_to_anchor=(0.980, 0.188),
    frameon=False,
    fontsize=12.6,
    handlelength=3.15,
    handletextpad=0.65,
    labelspacing=0.72,
)

fig.subplots_adjust(left=0.035, right=0.985, top=0.985, bottom=0.245)

for ext in ["svg", "pdf", "png", "tiff"]:
    output_path = FIG_BASE.with_suffix(f".{ext}")
    save_kwargs = {"bbox_inches": "tight", "facecolor": TOKENS["surface"]}
    if ext in {"png", "tiff"}:
        save_kwargs["dpi"] = 450
    if ext == "tiff":
        save_kwargs["pil_kwargs"] = {"compression": "tiff_lzw"}
    fig.savefig(output_path, **save_kwargs)

plt.show()
print(f"Theme-center nodes: {cluster_graph.number_of_nodes():,}")
print(f"Theme-center links: {cluster_graph.number_of_edges():,}")
print(f"Visible keyword nodes: {G.number_of_nodes():,}")
print(f"Visible keyword edges: {G.number_of_edges():,}")
print(f"Figure base path: {FIG_BASE}")

In [ ]:
# ===== 7. Write Method Log and Check Outputs =====
# The log follows the empirical topic-structure scope used in Table 1.

largest_theme = cluster_counts.idxmax()
largest_n = int(cluster_counts.max())
largest_share = round(largest_n / len(df) * 100, 1)

cluster_lines = []
for _, row in summary_df.iterrows():
    cluster_lines.append(
        f"- {row['topic_id']} {row['topic_label']}: N={row['n_records']} ({row['share_percent']}%); "
        f"keywords: {row['representative_keywords']}; objects: {row['main_built_environment_objects']}; "
        f"performance endpoints: {row['main_performance_endpoints']}."
    )

log_text = f"""# Subagent 10 Topic Clustering, Keyword Network, and Empirical Topic Matrix

Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## Input and Scope

- Input dataset: `data/{MAIN_DATASET_NAME}`
- Keyword table: `data/{KEYWORD_TABLE_NAME}` ({len(KEYWORD_TERMS)} terms)
- Records analyzed: {len(df):,}
- Matched keyword fields: `matched_keywords`, `matched_method_terms`, `matched_object_terms`, `matched_performance_terms`
- Unique matched terms used: {len(term_frequency)}
- Topic N check: {int(summary_df["n_records"].sum()):,} records across {len(summary_df)} clusters
- Complete weighted co-occurrence edges: {len(edge_df)}

## Topic Clusters

The rule-based dominant topic assignment uses interpretable BAE-oriented seed weights. Broad terms such as planning, big data, GIS, climate, and buildings are intentionally given lower cross-topic weights, while specific sensing, object, and performance terms receive stronger weights.

{chr(10).join(cluster_lines)}

Largest cluster: **{largest_theme}** with **{largest_n} records** ({largest_share}%).

## Network Figure

Fig10 uses the complete co-occurrence calculation but renders a single-panel publication figure for readability:

- Complete keyword co-occurrence edges retained in `10_keyword_edges.csv`: {len(edge_df)}
- Rendered theme-center nodes: {cluster_graph.number_of_nodes()}
- Rendered theme-center links: {cluster_graph.number_of_edges()}
- Rendered keyword nodes: {G.number_of_nodes()}
- Rendered keyword edges: {G.number_of_edges()}
- Keyword node size: record frequency.
- Keyword edge width: co-occurrence count.
- Node color: assigned topic cluster.
- All visible figure text is English.

## Table 1 Empirical Topic Matrix

`Table1_topic_summary_matrix.csv` reports empirical topic structure rather than the Fig. 3 coding framework. Columns are: `topic_id`, `topic_label`, `n_records`, `share_percent`, `representative_keywords`, `main_built_environment_objects`, `main_performance_endpoints`, and `empirical_interpretation`.

The built-environment object and performance endpoint columns are recalculated from `matched_object_terms` and `matched_performance_terms` in the 9222-record main table. Data-source, method, scale and validation/proxy framework columns are intentionally omitted to keep Table 1 distinct from the conceptual/coding framework.

## Draft Results Text

The keyword co-occurrence network reveals six empirical BAE topic clusters across the 2014-2026 deduplicated NSFC corpus (N={len(df):,}). The largest cluster is AI-enabled urban form and land-use measurement (N={int(summary_df.loc[summary_df['topic_id'] == 'T5', 'n_records'].iloc[0])}, {float(summary_df.loc[summary_df['topic_id'] == 'T5', 'share_percent'].iloc[0])}%), followed by nighttime light and carbon-energy performance (N={int(summary_df.loc[summary_df['topic_id'] == 'T3', 'n_records'].iloc[0])}, {float(summary_df.loc[summary_df['topic_id'] == 'T3', 'share_percent'].iloc[0])}%) and remote sensing and urban thermal environment (N={int(summary_df.loc[summary_df['topic_id'] == 'T1', 'n_records'].iloc[0])}, {float(summary_df.loc[summary_df['topic_id'] == 'T1', 'share_percent'].iloc[0])}%). The remaining clusters capture street-view and human-scale built environment analytics, exposure/health/environmental risk, and multisource urban resilience. Table 1 therefore presents the empirical topic structure: each cluster's size, representative keywords, dominant built-environment objects, dominant performance endpoints and result-oriented interpretation.

## Output Files

- `output/figures/Fig10_topic_network.svg`
- `output/figures/Fig10_topic_network.pdf`
- `output/figures/Fig10_topic_network.tiff`
- `output/figures/Fig10_topic_network.png`
- `output/tables/Table1_topic_summary_matrix.csv`
- `output/tables/10_keyword_edges.csv`
- `output/logs/10_topic_text.md`
"""

LOG_PATH.write_text(log_text, encoding="utf-8")

# Check output-file existence and minimum size.
expected_outputs = [
    FIG_BASE.with_suffix(".svg"),
    FIG_BASE.with_suffix(".pdf"),
    FIG_BASE.with_suffix(".tiff"),
    FIG_BASE.with_suffix(".png"),
    SUMMARY_TABLE_PATH,
    EDGE_TABLE_PATH,
    LOG_PATH,
]
missing_or_empty = [path for path in expected_outputs if (not path.exists()) or path.stat().st_size == 0]
if missing_or_empty:
    raise FileNotFoundError(f"Missing or empty outputs: {missing_or_empty}")

# Check that visible SVG text contains no Chinese characters, keeping all figure text in English.
svg_text = FIG_BASE.with_suffix(".svg").read_text(encoding="utf-8", errors="ignore")
if re.search(r"[一-鿿]", svg_text):
    raise ValueError("Fig10 SVG contains Chinese characters; visible figure text must be English.")

for path in expected_outputs:
    print(f"OK {path.relative_to(PROJECT_ROOT)} ({path.stat().st_size:,} bytes)")

display(Markdown(f"**Largest cluster:** {largest_theme} — {largest_n} records ({largest_share}%)."))
